In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import time
import tensorflow as tf
import tensorflow.python.keras
import matplotlib.pyplot as plt



KeyboardInterrupt



In [ ]:
from keras.applications import VGG19
from tensorflow import keras
from tensorflow.keras.layers import Flatten, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model
from glob import glob
from tensorflow.keras.optimizers import Adam

In [ ]:
IMAGE_SIZE = [224,224]

In [ ]:
# Create InceptionV3 Layer
vgg = VGG19(include_top=False,weights='imagenet',input_shape=IMAGE_SIZE + [3])
for layer in vgg.layers:
  layer.trainable = False

In [ ]:
# we will use the glob to see how many outputs / labels there are
folders = glob('./plantvillage/*')

In [ ]:
folders

In [ ]:
# Add Flatten and Dense in last layers
x = GlobalAveragePooling2D()(vgg.output)
x = Dropout(0.5)(x)
prediction = Dense(len(folders),activation='softmax')(x)

In [ ]:
model = Model(inputs=vgg.input,outputs = prediction)
plot_model(model)

In [ ]:
model.summary()

In [ ]:
# Compile the model that we have created with adam and the loss is categorical_crossentropy because the classification is more than 2 classes
model.compile(
    optimizer=Adam(learning_rate=1e-5), # Crucial for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


rotation_range=40: This will randomly rotate images by any degree between -40 and +40. This helps the model become invariant to the orientation of the objects in the images.

width_shift_range=0.2: This will randomly shift the image's width by a fraction of the total width. A value of 0.2 means the image can be shifted left or right by up to 20% of its width.

height_shift_range=0.2: Similar to width_shift_range, this shifts the image vertically by up to 20% of its height.

shear_range=0.2: This applies a shear transformation, which slants the shape of the image. It's like sliding one side of the image while keeping the other side fixed.

zoom_range=0.2: This will randomly zoom into the image by up to 20%. A zoom of [1 - 0.2, 1 + 0.2] is applied. So, it can zoom in or out.

horizontal_flip=True: This allows for the random flipping of images horizontally. This is useful when there's no assumption of asymmetry (e.g., a picture of a cat is still a cat if flipped).

fill_mode='nearest': When you rotate or shift an image, some pixels might be moved out of the frame, leaving empty space. This parameter tells the generator how to fill in those new pixels. 'nearest' simply uses the color of the nearest existing pixels.

In [ ]:
# Then use the CustomImageDataGenerator instead of ImageDataGenerator
train_datagen = ImageDataGenerator(
    validation_split=0.2,
    rescale=1./255,
    rotation_range=10,           # Very small rotations
    width_shift_range=0.05,      # Minimal shifts
    height_shift_range=0.05,
    shear_range=0.05,
    zoom_range=0.1,              # Small zoom
    horizontal_flip=True,
    fill_mode='nearest'
)


val_datagen = ImageDataGenerator(validation_split=0.2, rescale=1./255)
test_datagen = ImageDataGenerator(validation_split=0.2, rescale=1./255)

In [ ]:
# Load training and validation data using tf.data.Dataset
training_set = train_datagen.flow_from_directory(
    './plantvillage/',
    target_size=(224, 224),
    batch_size=8,
    class_mode='categorical',  # Make sure this is set to categorical
    subset='training',
    shuffle = True
)

val_set = val_datagen.flow_from_directory(
    './plantvillage/',
    target_size=(224, 224),
    batch_size=8,
    class_mode='categorical',  # Make sure this is set to categorical
    subset='validation'
)

test_dataset = test_datagen.flow_from_directory(
    './plantvillage/',
    target_size=(224, 224),
    batch_size = 8,
    class_mode = 'categorical',
    shuffle = False

)


In [ ]:
print(training_set)
print(val_set)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

In [ ]:
tic = time.perf_counter()
history = model.fit(training_set,
                    steps_per_epoch=len(training_set),
                    validation_data=val_set,
                    validation_steps=len(val_set),
                    epochs=40, callbacks=[early_stopping, reduce_lr])
# time
toc = time.perf_counter()
model.save('vgg19_test.keras')


In [ ]:
print("Total Time:{}".format(round((toc-tic)/60,2)))

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.plot(loss, label='train_loss')
plt.plot(val_loss,label = 'val_loss')
plt.legend()
plt.title("Train_loss: {:.4f}".format(history.history['loss'][-1]) +
          "\nValidation_loss: {:.4f}".format(history.history['val_loss'][-1]))
plt.show()

plt.plot(acc, label='train_acc')
plt.plot(val_acc,label = 'val_acc')
plt.title("Train_accuracy:{:.4f}".format(max(history.history['accuracy']))+
         "\nValidation_accuracy:{:.4f}".format(max(history.history['val_accuracy'])))
plt.legend()
plt.show()

In [ ]:
scores = model.evaluate(test_dataset)

In [ ]:
y_pred = model.predict(test_dataset)    

In [ ]:
y_predict = np.argmax(y_pred,axis=1)
y_predict

In [ ]:
y_true = test_dataset.classes
y_true

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
class_name = ['Tomato_Bacterial_spot','Tomato_Early_blight','Tomato_Late_blight','Tomato_Leaf_Mold','Tomato_Septoria_leaf_spot','Tomato_Spider_mites_Two_spotted_spider_mite','Tomato__Target_Spot','Tomato__Tomato_YellowLeaf__Curl_Virus','Tomato__Tomato_mosaic_virus','Tomato_healthy']

In [ ]:
print(classification_report(y_true,y_predict,target_names = class_name))

In [ ]:
import seaborn as sns

In [ ]:
cm = confusion_matrix(y_true,y_predict)
cm

In [ ]:
cm_normalized = np.round(cm/np.sum(cm,axis=1).reshape(-1,1),2)
print(cm_normalized)

In [ ]:
sns.heatmap(cm_normalized, annot=True,xticklabels=class_name, yticklabels=class_name,)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.show()

In [ ]:
import pickle 
# Save the history object to a file
with open('history_vgg19_2.pkl', 'wb') as file:
    pickle.dump(history.history, file)